In [1]:
from src.parameter_handler import ConfigManager, SimulationState
from omegaconf import OmegaConf
import numpy as np
from src.config import PathConfigs
from src.barrier_calibration import calibrate_wire_cutters
from src import helpers as hp
import src.barrier_calibration as bc
import xarray as xr

In [2]:
yamlpath = PathConfigs.PARAMETERS/"fixing_code.yaml"  

In [3]:
config = ConfigManager.initialize(yamlpath)
state = SimulationState(config)

geom = config.simulation.geometry
disorder = config.simulation.disorder
res = config.simulation.resolution
calib = config.simulation.barrier_calibration 

Vdis = state.Vdisx * 1.2

state.__dict__.keys()

mu = 2

Vz_var = state.params_list[:,2]
mu_var = state.params_list[:,1]

mu_set = list(set(mu_var))
Vz_set = list(set(Vz_var))

temps = [0,10,30,40]

state.target_GNs

Derived Hopping Parameter t: 16.5651
--- Running Simultaneous Cutter Barrier Calibration for targets: [0.01 0.05 0.1  0.14 0.19 0.23 0.28 0.32 0.35 0.41 0.45 0.49 0.54 0.58
 0.62 0.67 0.72 0.76 0.81 0.85 0.9 ] ---
Calibrated Symmetric Cutter Pairs [G_N, [UL, UR]]:
<xarray.DataArray (G_n: 21, side: 2)>
array([[14.26404595,  9.41165421],
       [ 7.28345699,  3.63172792],
       [ 4.94447281,  1.70830974],
       [ 3.92867994,  0.87011211],
       [ 3.06596357,  0.15286143],
       [ 2.55188799, -0.27899848],
       [ 2.04090884, -0.71367129],
       [ 1.70345565, -1.00506058],
       [ 1.4807735 , -1.19991567],
       [ 1.09401634, -1.54478989],
       [ 0.86966535, -1.7498173 ],
       [ 0.66605242, -1.94006269],
       [ 0.43512964, -2.16211984],
       [ 0.26582537, -2.33024344],
       [ 0.1079733 , -2.49234935],
       [-0.07584828, -2.68953492],
       [-0.2471092 , -2.88422261],
       [-0.37650393, -3.04099197],
       [-0.53024853, -3.24260005],
       [-0.64768124, -3.41316393

array([0.01, 0.05, 0.1 , 0.14, 0.19, 0.23, 0.28, 0.32, 0.35, 0.41, 0.45,
       0.49, 0.54, 0.58, 0.62, 0.67, 0.72, 0.76, 0.81, 0.85, 0.9 ])

In [9]:
state.calibrated_barriers.sel(G_n = 0.14, side = "left")

<xarray.DataArray ()>
array(3.92867994)
Coordinates:
    G_n      float64 0.14
    side     <U5 'left'

In [4]:
Gmatarr = hp.initialize_conductance_dataarray(
    target_temps=temps,
    G_n_values = state.target_GNs,
    mu_set=mu_set,
    Vz_set=Vz_set

)

Gmatarr.sel(temperature = 30, swept_barrier = "left", G_n = [0.01], chemical_potential = 0.2, zeeman_field = 0.6 )

KeyError: "not all values found in index 'chemical_potential'. Try setting the `method` keyword argument (example: method='nearest')."

In [ ]:
target_gn = 1.5

ul = bc.calibrate_single_barrier(
    target_gn=target_gn,
    side = "left",
    calibration_cfg=calib,
    geometry_cfg=geom,
    Vdisx=Vdis,
    t = state.t,
    alpha=state.alpha,
    mu_leads = state.t,
    mu_points = [mu]
)

ur = bc.calibrate_single_barrier(
    target_gn=target_gn,
    side = "right",
    calibration_cfg=calib,
    geometry_cfg=geom,
    Vdisx=Vdis,
    t = state.t,
    alpha=state.alpha,
    mu_leads = state.t,
    mu_points = [mu]
)

print(f"UL: {ul}    UR: {ur}")

In [ ]:
syst = hp.build_system_exact(
    t=state.t,
    mu=mu,
    gamma=geom.gamma,
    Delta0=geom.Delta_0,
    V_z=0.0,
    alpha=state.alpha,
    Ls=geom.Ls,
    Vdisx=Vdis,
    barrier_l=ul,
    barrier_r=ur,
    Ln = geom.Ln,
    Lb = geom.Lb
)

G_mat = hp.calc_conductance_matrix_exact(syst, E=0.5)

np.isclose(G_mat[0,0], G_mat[1,1], atol = 0.1) 


G_mat

In [ ]:
Gmats = []
asyms = np.linspace(-5, 5, 21)
ur_asyms = []

syst = hp.build_system_exact(
    t=state.t,
    mu=mu,
    gamma=geom.gamma,
    Delta0=geom.Delta_0,
    V_z=0.0,
    alpha=state.alpha,
    Ls=geom.Ls,
    Vdisx=Vdis,
    barrier_l=ul,
    barrier_r=ur,
    Ln = geom.Ln,
    Lb = geom.Lb
)

G_mat = hp.calc_conductance_matrix_exact(syst, E=0.0)

asyms

NameError: name 'np' is not defined

In [ ]:
syst = hp.build_system(
    t=state.t,
    mu=mu,
    gamma=geom.gamma,
    Delta0=geom.Delta_0,
    V_z=0.0,
    alpha=state.alpha,
    Ls=geom.Ls,
    Vdisx=Vdis,
    barrier_l=ul,
    barrier_r=ur,
    Ln = geom.Ln,
    Lb = geom.Lb,
    mu_leads=state.t

)

G_mat_renorm = hp.calc_conductance_matrix(syst, eng=0.0000000000)

all(np.isclose(G_mat_renorm, G_mat).flatten() )

In [ ]:
vdis_for_calib = state.Vdisx * float(disorder.V0) 
calibrated_cutters = bc.calibrate_wire_cutters(
    targets = np.asarray([0.2, 0.3]),
    calibration_cfg=calib,
    geometry_cfg=geom,
    Vdisx=vdis_for_calib,
    t=state.t,
    alpha=state.alpha,
    mu_leads=state.mu_leads,
)

In [ ]:
calibrated_cutters